In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    make_scorer
)

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/raw/online_shoppers_intention.csv")

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (12330, 18)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [9]:
X = df.drop(columns=["Revenue"])
y = df["Revenue"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("Train purchase rate:", y_train.mean())
print("Test purchase rate:", y_test.mean())


Train shape: (9864, 17)
Test shape: (2466, 17)
Train purchase rate: 0.15470397404703973
Test purchase rate: 0.1549067315490673


In [10]:
def add_engineered_features(data):
    data = data.copy()

    data["TotalPages"] = (
        data["Administrative"]
        + data["Informational"]
        + data["ProductRelated"]
    )

    data["TotalDuration"] = (
        data["Administrative_Duration"]
        + data["Informational_Duration"]
        + data["ProductRelated_Duration"]
    )

    data["AvgDurationPerPage"] = np.where(
        data["TotalPages"] > 0,
        data["TotalDuration"] / data["TotalPages"],
        0
    )

    data["ProductPageShare"] = np.where(
        data["TotalPages"] > 0,
        data["ProductRelated"] / data["TotalPages"],
        0
    )

    data["ProductDurationShare"] = np.where(
        data["TotalDuration"] > 0,
        data["ProductRelated_Duration"] / data["TotalDuration"],
        0
    )

    return data


X_train = add_engineered_features(X_train)
X_test = add_engineered_features(X_test)

engineered_features = [
    "TotalPages",
    "TotalDuration",
    "AvgDurationPerPage",
    "ProductPageShare",
    "ProductDurationShare"
]

print(X_train[engineered_features].head())


      TotalPages  TotalDuration  AvgDurationPerPage  ProductPageShare  \
4263          42    2078.005952           49.476332          0.809524   
5905           4     157.200000           39.300000          1.000000   
9434           4      42.000000           10.500000          1.000000   
3505          19    1543.566667           81.240351          0.894737   
2067          32     827.646212           25.863944          1.000000   

      ProductDurationShare  
4263              0.647619  
5905              1.000000  
9434              1.000000  
3505              0.781027  
2067              1.000000  


In [11]:
categorical_features = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType"
]

boolean_features = [
    "Weekend"
]

numeric_features = [
    col for col in X_train.columns
    if col not in categorical_features + boolean_features
]

print("Categorical features:")
print(categorical_features)

print("\nBoolean features:")
print(boolean_features)

print("\nNumber of numeric features:", len(numeric_features))
print(numeric_features)

Categorical features:
['Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType']

Boolean features:
['Weekend']

Number of numeric features: 15
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'TotalPages', 'TotalDuration', 'AvgDurationPerPage', 'ProductPageShare', 'ProductDurationShare']


In [12]:
preprocessor_scaled = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "bool",
            "passthrough",
            boolean_features
        )
    ]
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [13]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    make_scorer
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

print("5-fold Stratified Cross-Validation is ready.")

5-fold Stratified Cross-Validation is ready.


In [14]:
def summarize_cv(model_name, pipeline, X_data, y_data):

    scores = cross_validate(
        pipeline,
        X_data,
        y_data,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=-1
    )

    row = {
        "Model": model_name
    }

    for metric in scoring:
        row[f"CV_{metric}_mean"] = scores[f"test_{metric}"].mean()
        row[f"CV_{metric}_std"] = scores[f"test_{metric}"].std()

    row["Train_F1_mean"] = scores["train_f1"].mean()
    row["Validation_F1_mean"] = scores["test_f1"].mean()

    return row, scores


print("Evaluation function created successfully.")

Evaluation function created successfully.


In [15]:
lr_baseline = Pipeline([
    (
        "preprocessor",
        preprocessor_scaled
    ),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

lr_baseline_row, lr_baseline_scores = summarize_cv(
    "Logistic Regression - Baseline",
    lr_baseline,
    X_train,
    y_train
)

pd.DataFrame([lr_baseline_row])

0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0

,Model,CV_accuracy_mean,CV_accuracy_std,CV_precision_mean,CV_precision_std,CV_recall_mean,CV_recall_std,CV_f1_mean,CV_f1_std,CV_roc_auc_mean,CV_roc_auc_std,CV_pr_auc_mean,CV_pr_auc_std,Train_F1_mean,Validation_F1_mean
0,Logistic Regression - Baseline,0.884733,0.00521,0.742051,0.026375,0.39055,0.030803,0.511226,0.030538,0.894717,0.005789,0.645259,0.017388,0.518013,0.511226


In [16]:
lr_balanced = Pipeline([
    (
        "preprocessor",
        preprocessor_scaled
    ),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )
])

lr_balanced_row, lr_balanced_scores = summarize_cv(
    "Logistic Regression - Balanced",
    lr_balanced,
    X_train,
    y_train
)

pd.DataFrame([
    lr_baseline_row,
    lr_balanced_row
])[
    [
        "Model",
        "CV_accuracy_mean",
        "CV_precision_mean",
        "CV_recall_mean",
        "CV_f1_mean",
        "CV_roc_auc_mean",
        "CV_pr_auc_mean"
    ]
]

0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0

,Model,CV_accuracy_mean,CV_precision_mean,CV_recall_mean,CV_f1_mean,CV_roc_auc_mean,CV_pr_auc_mean
0,Logistic Regression - Baseline,0.884733,0.742051,0.390550,0.511226,0.894717,0.645259
1,Logistic Regression - Balanced,0.851987,0.515047,0.764082,0.615193,0.905667,0.656848


In [17]:
lr_tune_pipeline = Pipeline([
    (
        "preprocessor",
        preprocessor_scaled
    ),
    (
        "model",
        LogisticRegression(
            max_iter=3000,
            solver="liblinear",
            random_state=42
        )
    )
])

lr_param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__penalty": ["l1", "l2"],
    "model__class_weight": [None, "balanced"]
}

lr_search = GridSearchCV(
    estimator=lr_tune_pipeline,
    param_grid=lr_param_grid,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    refit=True
)

lr_search.fit(X_train, y_train)

print("Best CV PR-AUC:", lr_search.best_score_)
print("Best parameters:", lr_search.best_params_)

0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.01s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0

Best CV PR-AUC: 0.6664949989603146
Best parameters: {'model__C': 0.01, 'model__class_weight': 'balanced', 'model__penalty': 'l1'}


/Users/praseethanethranjanalahandasinghe/Documents/BuySense/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/praseethanethranjanalahandasinghe/Documents/BuySense/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


In [18]:
best_lr = lr_search.best_estimator_

lr_tuned_row, lr_tuned_scores = summarize_cv(
    "Logistic Regression - Tuned",
    best_lr,
    X_train,
    y_train
)

pd.DataFrame([
    lr_baseline_row,
    lr_balanced_row,
    lr_tuned_row
])[
    [
        "Model",
        "CV_precision_mean",
        "CV_recall_mean",
        "CV_f1_mean",
        "CV_roc_auc_mean",
        "CV_pr_auc_mean"
    ]
]

0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0

,Model,CV_precision_mean,CV_recall_mean,CV_f1_mean,CV_roc_auc_mean,CV_pr_auc_mean
0,Logistic Regression - Baseline,0.742051,0.390550,0.511226,0.894717,0.645259
1,Logistic Regression - Balanced,0.515047,0.764082,0.615193,0.905667,0.656848
2,Logistic Regression - Tuned,0.583025,0.752300,0.656666,0.911277,0.666495
